# LangSmith Multi-Agent Trace Evaluation

This notebook is self-contained: it stores five synthetic traces, logs them to LangSmith, retrieves all root traces created within a configurable time window, evaluates each trace with six LLM-based metrics, and exports the results to JSON and Excel.

Run the cells from top to bottom. Re-running the logging cell publishes another set of five traces.


## 1. Install Dependencies

Installs the packages required for LangSmith trace logging and retrieval, Azure OpenAI evaluation, and Excel export.


In [ ]:
!pip install -q -U \
    langsmith \
    openai \
    pandas==2.2.2 \
    openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.3/661.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.1 MB/s eta 0:00:00


## 2. Configure Clients

Imports the required libraries and initializes the LangSmith and Azure OpenAI clients used throughout the notebook.


In [ ]:
import os
import json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd

from langsmith import Client
from langsmith.run_trees import RunTree
from openai import AzureOpenAI


# ============================================================
# LangSmith Configuration
# ============================================================

LANGSMITH_API_KEY = "lsv2_f24c_90788ae41c"
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
LANGSMITH_PROJECT = "multi-agent-demo"

os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_ENDPOINT"] = LANGSMITH_ENDPOINT
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT

langsmith_client = Client()


# ============================================================
# Azure OpenAI Configuration
# ============================================================

AZURE_OPENAI_ENDPOINT = "https://trsion=2025-01-01-preview"
AZURE_OPENAI_API_KEY = "5xD6IeVNWL7eYFXJ3w3AAABACOGD59K"
AZURE_OPENAI_API_VERSION = "20eview"
AZURE_OPENAI_DEPLOYMENT = "gpano"

azure_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("✓ LangSmith client initialized")
print("✓ Azure OpenAI client initialized")


✓ LangSmith client initialized
✓ Azure OpenAI client initialized


## 3. Embedded Trace Data and Logging

Loads the baseline trace and four evaluation variants directly from the notebook. Run this cell to publish all five traces to the configured LangSmith project. Set `LOG_TRACES = False` to skip publishing and evaluate traces already stored in LangSmith instead.


In [ ]:
# ============================================================
# Embedded Trace Data (detailed, plain-English, no abbreviations)
# ============================================================
# One realistic "good" run (baseline) is authored in full, then each faulty
# variant is a deep copy with a single deliberate defect re-applied:
#   variant_A : wrong / partial extracted parameters -> a bad plan
#   variant_B : steps run in the wrong order, steps missing, a rogue tool
#   variant_C : the written report is not supported by the gathered evidence
#   variant_D : the final answer does not actually answer the question
# The specialist "tools" are the sub-agents; the per-trace expected tool schema
# and trajectory live in the expected-config Excel (see the expected-config cell).

import copy


def node(name, run_type, inputs, outputs, children=None, duration=0.6,
         tags=None, metadata=None):
    """Helper to build one span (LangSmith run) with a consistent shape."""
    return {
        "name": name,
        "run_type": run_type,
        "duration": duration,
        "tags": tags or [],
        "metadata": metadata or {},
        "inputs": inputs,
        "outputs": outputs,
        "children": children or [],
    }


# ---- Content reused across spans (kept consistent so evidence lines up) ----
USER_REQUEST = (
    "We are a mid-sized automobile manufacturer based in the United States. "
    "Please analyze the competitive landscape for electric vehicles in Europe "
    "and recommend a market-entry strategy for the year 2027."
)

EXTRACTED_PARAMETERS = {
    "company_profile": {
        "size": "mid-sized",
        "home_country": "United States",
        "industry": "automobile manufacturing",
    },
    "market": {
        "product_type": "electric vehicles",
        "region": "Europe",
    },
    "target_year": 2027,
    "objective": "Recommend a market-entry strategy",
    "deliverables": [
        "A competitive landscape analysis of the European electric vehicle market",
        "A recommended market-entry strategy for the year 2027",
    ],
}

PLANNING_CONTEXT = {
    "planned_region": "Europe",
    "planned_product": "electric vehicles",
    "planned_year": 2027,
    "deliverables": EXTRACTED_PARAMETERS["deliverables"],
}

SUPERVISOR_RULES = 'You are the Supervisor Agent for a multi-agent market-analysis workflow. Parameter extraction has already produced the structured planning parameters. Your job now is to produce a clear, numbered execution PLAN that routes the specialist agents in the correct order, and to name the agent that runs first.\n\nThe plan MUST route the specialist agents in exactly this order:\n1. **Research Agent**: gather market evidence. It MAY invoke ONLY these sub-agents: Web Search Sub-Agent, Document Retrieval Sub-Agent, Source Validator Sub-Agent.\n2. **Analysis Agent**: size the opportunity and benchmark competitors. It MAY invoke ONLY these sub-agents: Market Sizing Sub-Agent, Competitor Benchmarking Sub-Agent.\n3. **Report Writer Agent**: turn the findings into a market-entry recommendation. It MAY invoke ONLY these sub-agents: Drafting Sub-Agent, Formatting & Citation Sub-Agent.\n\nSupervisor-internal gating (the supervisor performs these automatically BETWEEN the numbered stages; they are NOT plan steps and must not be listed as steps in the plan): after each specialist stage the supervisor reviews the result before advancing, and it may send the Research Agent back once for a regulatory addendum if a gap remains.\n\nRouting and ordering rules (mandatory):\n- The plan MUST begin by routing to the Research Agent.\n- Research MUST come before Analysis, and Analysis MUST come before Report Writing.\n- NEVER route to the Report Writer Agent before research and analysis are complete.\n- A specialist agent MUST NOT invoke any tool or sub-agent other than the ones listed for it above; doing so is a violation.\n- The plan MUST address the product type, market region, and target year stated by the customer.'


RESEARCH_SUMMARY = (
    "The European electric vehicle market is large and growing quickly. Germany "
    "is the biggest market by the number of vehicles sold, while Norway and the "
    "Netherlands have the highest share of electric vehicles among new cars. The "
    "public charging network is strongest in Germany and the Netherlands. Every "
    "figure was checked against two independent industry sources and was consistent."
)

REGULATORY_SUMMARY = (
    "European Union regulations are becoming more favorable for electric vehicles. "
    "Carbon dioxide emission targets for car fleets are tightening between 2025 and "
    "2030, which pushes manufacturers to sell more electric vehicles. In addition, "
    "purchase incentives are increasingly tied to using battery packs assembled "
    "inside the European Union, and countries such as Hungary and Poland offer "
    "financial incentives for building electric vehicle and battery factories."
)

ANALYSIS_FINDING = (
    "There is room in the mid-priced compact-car segment, at around 38,000 euros, "
    "especially for a company that pairs a competitive price with a strong "
    "charging-network partnership. MG Motor currently owns the low-price end of the "
    "market, while Volkswagen and Tesla dominate the premium and upper-middle range. "
    "A well-priced compact electric car with reliable charging access would sit in a "
    "currently underserved gap in the market."
)

FINAL_REPORT = (
    "Recommended market-entry strategy for the European Union in 2027. "
    "First, enter through Germany and the Netherlands, because the research shows they "
    "have the highest electric vehicle adoption and the strongest public charging networks. "
    "Second, position the vehicle in the mid-priced compact-car segment at around 38,000 "
    "euros, below the premium and upper-middle range held by Volkswagen and Tesla and above "
    "the low-price end held by MG Motor, which the analysis identifies as an underserved gap. "
    "Third, pair the vehicle with a strong charging-network partnership so customers have "
    "reliable fast-charging access, since the analysis found this is what makes the mid-priced "
    "gap winnable. "
    "Fourth, assemble the battery packs inside the European Union, for example in Hungary or "
    "Poland, to qualify for the local-content purchase incentives and factory incentives "
    "described in the regulatory findings. "
    "Finally, time the launch to the tightening European Union carbon dioxide fleet targets "
    "between 2025 and 2030, which are pushing manufacturers to sell more electric vehicles."
)


# ============================================================
# BASELINE — a clean, correct run
# ============================================================
def build_baseline():
    parameter_extraction = node(
        "Parameter Extraction Agent", "llm",
        inputs={
            "system_instruction": (
                "Read the customer's request carefully and pull out the key planning "
                "details as clearly labeled fields, so the rest of the team has an "
                "accurate brief to work from."
            ),
            "user_request": USER_REQUEST,
        },
        outputs={
            "role": "assistant",
            "parameters": copy.deepcopy(EXTRACTED_PARAMETERS),
            "extraction_notes": (
                "Every field was stated directly by the customer in the request; "
                "nothing was guessed or inferred."
            ),
        },
    )

    plan_and_route = node(
        "Supervisor: Plan & Route", "llm",
        inputs={
            "planning_parameters": copy.deepcopy(EXTRACTED_PARAMETERS),
            "instruction": SUPERVISOR_RULES,
        },
        outputs={
            "role": "assistant",
            "plan": (
                "Step 1: Send the Research Agent to gather evidence on the European "
                "electric vehicle market, including sales, adoption by country, and the "
                "charging network. "
                "Step 2: Send the Analysis Agent to estimate the size of the opportunity "
                "and compare the main competitors. "
                "Step 3: Send the Report Writer Agent to turn the findings into a clear "
                "market-entry recommendation. "
                "Begin by routing to the Research Agent."
            ),
            "route_to": "Research Agent",
            "planning_context": copy.deepcopy(PLANNING_CONTEXT),
        },
    )

    web_search_market = node(
        "Web Search Sub-Agent", "tool",
        inputs={
            "tool_name": "web_search",
            "search_query": "Europe electric vehicle sales by country 2025 and 2026 market share",
            "purpose": (
                "Find recent figures on how many electric vehicles are sold in Europe and "
                "which countries lead in adoption."
            ),
        },
        outputs={
            "results": [
                {
                    "finding": (
                        "Battery electric vehicles made up roughly 16 percent of all new "
                        "cars sold in the European Union in 2025."
                    ),
                    "source_name": "European Automobile Manufacturers Association",
                },
                {
                    "finding": (
                        "In Norway, more than 80 percent of new cars sold are fully electric."
                    ),
                    "source_name": "Norwegian Road Federation",
                },
                {
                    "finding": (
                        "The largest manufacturers selling electric cars in Europe are Tesla, "
                        "the Volkswagen Group, Stellantis, BMW, and Hyundai-Kia."
                    ),
                    "source_name": "Industry sales tracker",
                },
                {
                    "finding": (
                        "Germany sells the most electric vehicles by total volume, while "
                        "Norway and the Netherlands have the highest share of electric "
                        "vehicles among new cars."
                    ),
                    "source_name": "European Automobile Manufacturers Association",
                },
            ]
        },
    )

    document_retrieval = node(
        "Document Retrieval Sub-Agent", "retriever",
        inputs={
            "search_query": "European Union public charging infrastructure density 2026 report",
            "document_type": "industry report",
            "purpose": (
                "Find detailed data on how many public charging points exist in Europe and "
                "where the network is strongest."
            ),
        },
        outputs={
            "documents": [
                {
                    "title": "European Union Electric Vehicle Charging Infrastructure Report 2026",
                    "publisher": "European Automobile Manufacturers Association",
                    "year": 2026,
                    "page": 12,
                    "passage": (
                        "As of early 2026 there are approximately 630,000 public charging "
                        "points across the European Union, up from about 480,000 a year "
                        "earlier. Most of the growth is concentrated in Germany, the "
                        "Netherlands, and France."
                    ),
                    "why_relevant": (
                        "Gives an overall count of public charging points and shows which "
                        "countries are expanding their networks fastest."
                    ),
                },
                {
                    "title": "Global Electric Vehicle Outlook",
                    "publisher": "International Energy Agency",
                    "year": 2026,
                    "page": 47,
                    "passage": (
                        "The Netherlands and Germany lead the European Union in the rollout "
                        "of fast chargers. The Netherlands has the highest number of charging "
                        "points per person of any large country in Europe."
                    ),
                    "why_relevant": (
                        "Confirms that the Netherlands and Germany have the strongest "
                        "fast-charging networks in the region."
                    ),
                },
            ]
        },
    )

    source_validator = node(
        "Source Validator Sub-Agent", "chain",
        inputs={
            "claims_to_verify": [
                "Battery electric vehicles were about 16 percent of new car sales in the "
                "European Union in 2025.",
                "There are roughly 630,000 public charging points in the European Union.",
            ],
            "verification_method": (
                "Compare each claim against at least two independent industry sources and "
                "flag any disagreement."
            ),
        },
        outputs={
            "all_claims_validated": True,
            "details": [
                {
                    "claim": (
                        "Battery electric vehicles were about 16 percent of new car sales in "
                        "the European Union in 2025."
                    ),
                    "status": "confirmed",
                    "checked_against": [
                        "European Automobile Manufacturers Association",
                        "International Energy Agency",
                    ],
                },
                {
                    "claim": (
                        "There are roughly 630,000 public charging points in the European Union."
                    ),
                    "status": "confirmed",
                    "checked_against": ["European Automobile Manufacturers Association"],
                },
            ],
            "notes": (
                "Both figures matched across the sources that were checked, so they can be "
                "relied on in the analysis."
            ),
        },
    )

    research_agent = node(
        "Research Agent", "chain",
        inputs={
            "planning_context": copy.deepcopy(PLANNING_CONTEXT),
            "task": (
                "Gather the evidence the plan calls for: how large the European electric "
                "vehicle market is, which countries lead in adoption, and how strong the "
                "public charging network is."
            ),
        },
        outputs={
            "summary": RESEARCH_SUMMARY,
            "confidence": 0.82,
            "alignment_with_plan": (
                "This research covers the European electric vehicle market for 2027 exactly "
                "as the planner requested."
            ),
        },
        children=[web_search_market, document_retrieval, source_validator],
    )

    review_research = node(
        "Supervisor: Review Research", "llm",
        inputs={
            "question_for_reviewer": (
                "Is the research complete and reliable enough to move on to analysis?"
            ),
            "research_summary": RESEARCH_SUMMARY,
        },
        outputs={
            "role": "assistant",
            "decision": (
                "The research is solid and well sourced. It covers market size, "
                "country-by-country adoption, and the charging network, so it is enough to "
                "begin the analysis."
            ),
            "route_to": "Analysis Agent",
        },
    )

    market_sizing = node(
        "Market Sizing Sub-Agent", "tool",
        inputs={
            "tool_name": "market_sizing_model",
            "parameters": {
                "region": "European Union",
                "vehicle_segment": "compact battery electric cars",
                "year": 2027,
            },
        },
        outputs={
            "total_addressable_market_units_2027": 2100000,
            "serviceable_market_units_2027": 340000,
            "average_selling_price_euros": 38500,
            "explanation": (
                "About 2.1 million compact electric cars are expected to be sold across the "
                "European Union in 2027. Of those, roughly 340,000 are realistically "
                "reachable for a new entrant in the first few years. The average selling "
                "price in this segment is about 38,500 euros."
            ),
        },
    )

    competitor_benchmarking = node(
        "Competitor Benchmarking Sub-Agent", "chain",
        inputs={
            "competitors_to_compare": [
                "Volkswagen ID.3 and ID.4",
                "Tesla Model 3 and Model Y",
                "Renault",
                "MG Motor",
            ],
            "comparison_dimensions": [
                "price", "brand strength", "charging access", "market position",
            ],
        },
        outputs={
            "positioning": [
                {"competitor": "Volkswagen ID.3", "approximate_price_euros": 40000,
                 "strength": "trusted, well-established brand"},
                {"competitor": "Tesla Model 3", "approximate_price_euros": 42000,
                 "strength": "technology leadership and strong brand image"},
                {"competitor": "MG Motor", "approximate_price_euros": 30000,
                 "strength": "lowest price in the market"},
            ],
            "identified_gap": (
                "The mid-priced compact segment, paired with a strong charging-network "
                "partnership, is currently underserved."
            ),
        },
    )

    analysis_agent = node(
        "Analysis Agent", "chain",
        inputs={
            "planning_context": copy.deepcopy(PLANNING_CONTEXT),
            "research_summary": RESEARCH_SUMMARY,
            "task": (
                "Using the research findings, estimate the size of the opportunity and "
                "compare the main competitors so we can find a gap in the market."
            ),
        },
        outputs={"finding": ANALYSIS_FINDING, "confidence": 0.7},
        children=[market_sizing, competitor_benchmarking],
    )

    review_analysis = node(
        "Supervisor: Review Analysis", "llm",
        inputs={
            "question_for_reviewer": (
                "Is the analysis complete enough to start writing the recommendation?"
            ),
            "analysis_finding": ANALYSIS_FINDING,
        },
        outputs={
            "role": "assistant",
            "decision": (
                "The analysis is missing the European regulatory picture, such as carbon "
                "dioxide emission targets and rules that reward assembling battery packs "
                "inside the European Union. Send the Research Agent back to gather this "
                "before writing the report."
            ),
            "gap_identified": (
                "No coverage of European Union regulations and purchase incentives."
            ),
            "route_to": "Research Agent",
        },
    )

    web_search_regulation = node(
        "Web Search Sub-Agent", "tool",
        inputs={
            "tool_name": "web_search",
            "search_query": (
                "European Union 2027 carbon dioxide fleet targets and local battery content "
                "subsidy rules"
            ),
            "purpose": (
                "Find the regulations and incentives that will shape electric vehicle "
                "strategy in 2027."
            ),
        },
        outputs={
            "results": [
                {
                    "finding": (
                        "European Union carbon dioxide targets for car fleets tighten "
                        "between 2025 and 2030, pushing manufacturers to increase the share "
                        "of electric vehicles they sell."
                    ),
                    "source_name": "European Commission",
                },
                {
                    "finding": (
                        "Purchase incentives are increasingly linked to using battery packs "
                        "assembled inside the European Union."
                    ),
                    "source_name": "European policy tracker",
                },
                {
                    "finding": (
                        "Hungary and Poland offer financial incentives for building electric "
                        "vehicle and battery factories."
                    ),
                    "source_name": "Regional investment agencies",
                },
            ]
        },
    )

    research_addendum = node(
        "Research Agent", "chain",
        inputs={
            "task": (
                "Additional research: European Union regulations for electric vehicles in "
                "2027, including carbon dioxide emission targets and rules that reward using "
                "battery packs built inside the European Union."
            )
        },
        outputs={"summary": REGULATORY_SUMMARY, "confidence": 0.86},
        children=[web_search_regulation],
    )

    review_addendum = node(
        "Supervisor: Review Addendum", "llm",
        inputs={
            "question_for_reviewer": (
                "Do we now have enough, including the regulatory picture, to write the "
                "recommendation?"
            ),
            "addendum_summary": REGULATORY_SUMMARY,
        },
        outputs={
            "role": "assistant",
            "decision": (
                "The picture is now complete, including regulations and incentives. Send the "
                "Report Writer Agent to produce the final recommendation."
            ),
            "route_to": "Report Writer Agent",
        },
    )

    drafting = node(
        "Drafting Sub-Agent", "llm",
        inputs={
            "messages": [
                {"role": "system",
                 "content": (
                     "Write a clear and concise market-entry recommendation that a business "
                     "audience can act on."
                 )},
                {"role": "user",
                 "content": (
                     "Combine the research and analysis into a market-entry plan for the "
                     "European Union in 2027."
                 )},
            ]
        },
        outputs={"role": "assistant", "content": FINAL_REPORT},
    )

    formatting = node(
        "Formatting & Citation Sub-Agent", "chain",
        inputs={
            "draft_text": FINAL_REPORT,
            "sources_to_cite": [
                "European Union Electric Vehicle Charging Infrastructure Report 2026",
                "Global Electric Vehicle Outlook",
            ],
        },
        outputs={
            "formatting_applied": True,
            "citations_added": 2,
            "final_report": FINAL_REPORT,
        },
    )

    report_writer = node(
        "Report Writer Agent", "chain",
        inputs={
            "planning_context": copy.deepcopy(PLANNING_CONTEXT),
            "analysis_summary": ANALYSIS_FINDING,
            "research_summary": RESEARCH_SUMMARY,
            "regulatory_summary": REGULATORY_SUMMARY,
            "task": (
                "Turn all of the findings into a clear, well-supported market-entry "
                "recommendation for the year 2027."
            ),
        },
        outputs={"report": FINAL_REPORT, "word_count": 118},
        children=[drafting, formatting],
    )

    final_review = node(
        "Supervisor: Final Review", "llm",
        inputs={
            "planning_context": copy.deepcopy(PLANNING_CONTEXT),
            "final_report": FINAL_REPORT,
            "question_for_reviewer": (
                "Does the final report fully answer the customer's request with supporting "
                "evidence?"
            ),
        },
        outputs={
            "role": "assistant",
            "decision": (
                "Approved. The report answers the customer's question with supporting data "
                "and includes the regulatory context."
            ),
            "approved": True,
        },
    )

    return node(
        "Supervisor Agent", "chain",
        inputs={"user_request": USER_REQUEST},
        outputs={
            "final_answer": FINAL_REPORT,
            "iterations": 2,
            "agents_used": ["Research Agent", "Analysis Agent", "Report Writer Agent"],
            "approved": True,
        },
        children=[
            parameter_extraction, plan_and_route,
            research_agent, review_research,
            analysis_agent, review_analysis,
            research_addendum, review_addendum,
            report_writer, final_review,
        ],
    )


def _child(root, name):
    """First top-level child span with the given name."""
    return next(c for c in root["children"] if c["name"] == name)


# ============================================================
# VARIANT A — wrong / partial extracted parameters -> bad plan
# ============================================================
def build_variant_a():
    root = build_baseline()
    root["name"] = "Supervisor Agent — variant_A_poor_parameter_and_plan"

    wrong_parameters = {
        "company_profile": {"home_country": "Europe"},
        "market": {"product_type": "hybrid cars", "region": "North America"},
        "target_year": 2025,
        "objective": "Prepare a general product overview",
    }
    pe = _child(root, "Parameter Extraction Agent")
    pe["outputs"] = {
        "role": "assistant",
        "parameters": wrong_parameters,
        "extraction_notes": (
            "Only partial details were captured, and several were guessed rather than taken "
            "from the customer's request. The home country, product type, region, and year "
            "do not match what the customer actually said."
        ),
    }

    plan = _child(root, "Supervisor: Plan & Route")
    plan["inputs"]["planning_parameters"] = copy.deepcopy(wrong_parameters)
    plan["outputs"] = {
        "role": "assistant",
        "plan": (
            "Step 1: Prepare a general overview of hybrid cars in North America for 2025. "
            "Step 2: Write a short product note. Route directly to the Report Writer Agent."
        ),
        "route_to": "Report Writer Agent",
        "planning_context": {
            "planned_region": "North America",
            "planned_product": "hybrid cars",
            "planned_year": 2025,
            "deliverables": ["A short product note"],
        },
    }
    return root


# ============================================================
# VARIANT B — wrong order, missing steps, and a rogue tool
# ============================================================
def build_variant_b():
    base = build_baseline()
    root = node(
        "Supervisor Agent — variant_B_poor_trajectory_and_tools", "chain",
        inputs={"user_request": USER_REQUEST},
        outputs=copy.deepcopy(base["outputs"]),
    )

    currency_conversion = node(
        "Currency Conversion Sub-Agent", "tool",
        inputs={
            "tool_name": "currency_converter",
            "amount_to_convert": 38500,
            "from_currency": "euros",
            "to_currency": "United States dollars",
        },
        outputs={
            "converted_amount": 41000,
            "exchange_rate_used": 1.065,
            "note": (
                "Converted the average selling price from euros into United States dollars. "
                "This tool is not part of the approved research, analysis, or writing "
                "workflow."
            ),
        },
    )

    # Report writing happens BEFORE any research or analysis, several review and
    # sub-agent steps are missing, and an unexpected currency tool is invoked.
    report_writer = copy.deepcopy(_child(base, "Report Writer Agent"))
    analysis = copy.deepcopy(_child(base, "Analysis Agent"))
    analysis["children"] = [c for c in analysis["children"]
                            if c["name"] == "Market Sizing Sub-Agent"]
    research = copy.deepcopy(_child(base, "Research Agent"))
    research["children"] = [c for c in research["children"]
                            if c["name"] == "Web Search Sub-Agent"]

    root["children"] = [
        copy.deepcopy(_child(base, "Parameter Extraction Agent")),
        copy.deepcopy(_child(base, "Supervisor: Plan & Route")),
        report_writer,
        currency_conversion,
        analysis,
        research,
        copy.deepcopy(_child(base, "Supervisor: Final Review")),
    ]
    return root


# ============================================================
# VARIANT C — report not supported by the gathered evidence
# ============================================================
def build_variant_c():
    root = build_baseline()
    root["name"] = "Supervisor Agent — variant_C_poor_summary_accuracy"

    unsupported_report = (
        "Recommended strategy: immediately launch a fleet of fully self-flying electric "
        "vehicles across every country in Europe in 2027, funded by a guaranteed ten "
        "billion euro grant from the European Union. Price every vehicle at exactly "
        "12,000 euros, buy the Tesla company outright, and build the world's largest "
        "battery factory on the Moon before the end of the year. Charging will not be "
        "needed because the vehicles recharge themselves while driving."
    )

    report_writer = _child(root, "Report Writer Agent")
    report_writer["outputs"] = {"report": unsupported_report, "word_count": 92}
    drafting = next(c for c in report_writer["children"]
                    if c["name"] == "Drafting Sub-Agent")
    drafting["outputs"] = {"role": "assistant", "content": unsupported_report}
    formatting = next(c for c in report_writer["children"]
                      if c["name"] == "Formatting & Citation Sub-Agent")
    formatting["outputs"] = {
        "formatting_applied": True, "citations_added": 2, "final_report": unsupported_report,
    }
    _child(root, "Supervisor: Final Review")["inputs"]["final_report"] = unsupported_report
    return root


# ============================================================
# VARIANT D — final answer does not actually answer the question
# ============================================================
def build_variant_d():
    root = build_baseline()
    root["name"] = "Supervisor Agent — variant_D_poor_goal_accomplishment"

    vague_answer = (
        "Europe is an important market for electric vehicles. The company should keep "
        "researching the market and consider expanding at some point in the future when "
        "the timing seems right."
    )

    root["outputs"] = {
        "final_answer": vague_answer,
        "iterations": 2,
        "agents_used": ["Research Agent", "Analysis Agent", "Report Writer Agent"],
        "approved": True,
    }
    report_writer = _child(root, "Report Writer Agent")
    report_writer["outputs"] = {"report": vague_answer, "word_count": 34}
    drafting = next(c for c in report_writer["children"]
                    if c["name"] == "Drafting Sub-Agent")
    drafting["outputs"] = {"role": "assistant", "content": vague_answer}
    formatting = next(c for c in report_writer["children"]
                      if c["name"] == "Formatting & Citation Sub-Agent")
    formatting["outputs"] = {
        "formatting_applied": True, "citations_added": 0, "final_report": vague_answer,
    }
    final_review = _child(root, "Supervisor: Final Review")
    final_review["inputs"]["final_report"] = vague_answer
    final_review["outputs"] = {
        "role": "assistant",
        "decision": "Approved: a short, general market note was prepared.",
        "approved": True,
    }
    return root


EMBEDDED_TRACE_DATA = {
    "project": "multi-agent-market-entry",
    "traces": [
        {"id": "baseline", "trace": build_baseline()},
        {"id": "variant_A_poor_parameter_and_plan", "trace": build_variant_a()},
        {"id": "variant_B_poor_trajectory_and_tools", "trace": build_variant_b()},
        {"id": "variant_C_poor_summary_accuracy", "trace": build_variant_c()},
        {"id": "variant_D_poor_goal_accomplishment", "trace": build_variant_d()},
    ],
}


# Tag each root input with its scenario id so evaluators can map the per-trace
# expected trajectory/tools by inspecting the ROOT trace input (see expected-config cell).
for _record in EMBEDDED_TRACE_DATA["traces"]:
    _record["trace"]["inputs"]["scenario"] = _record["id"]


print(f"Loaded {len(EMBEDDED_TRACE_DATA['traces'])} embedded trace(s).")


# ============================================================
# Log Embedded Traces to LangSmith
# ============================================================

class VirtualClock:
    """Creates increasing timestamps without waiting in real time."""

    def __init__(self, start_time):
        self.current_time = start_time

    def now(self):
        timestamp = self.current_time
        self.current_time += timedelta(milliseconds=50)
        return timestamp

    def advance(self, seconds):
        self.current_time += timedelta(seconds=float(seconds))
        return self.current_time


def emit_trace_node(node, parent, clock, project, client):
    extra = {"metadata": node["metadata"]} if node.get("metadata") else {}
    run_arguments = {
        "name": node["name"],
        "run_type": node["run_type"],
        "inputs": node.get("inputs", {}),
        "start_time": clock.now(),
        "tags": node.get("tags", []),
        "extra": extra,
    }

    run = (
        RunTree(project_name=project, client=client, **run_arguments)
        if parent is None
        else parent.create_child(**run_arguments)
    )
    run.post()

    for child in node.get("children", []):
        emit_trace_node(child, run, clock, project, client)

    run.end(
        outputs=node.get("outputs", {}),
        end_time=clock.advance(node.get("duration", 0.5)),
    )
    run.patch()
    return run


def log_embedded_traces(trace_data=EMBEDDED_TRACE_DATA, project_name=LANGSMITH_PROJECT):
    """Publish every embedded trace and return its identifier and URL."""
    records = trace_data.get("traces", [])
    if not records:
        raise ValueError("Embedded trace data does not contain any traces.")

    logged_traces = []
    for record in records:
        root = emit_trace_node(
            record["trace"],
            parent=None,
            clock=VirtualClock(datetime.now(timezone.utc)),
            project=project_name,
            client=langsmith_client,
        )
        try:
            url = root.get_url()
        except Exception:
            url = None
        logged_traces.append({"variant_id": record["id"], "trace_id": str(root.id), "url": url})

    langsmith_client.flush()
    return logged_traces


LOG_TRACES = True

if LOG_TRACES:
    logged_traces = log_embedded_traces()
    print(f"Logged {len(logged_traces)} trace(s) to '{LANGSMITH_PROJECT}'.")
    for item in logged_traces:
        print(f"- {item['variant_id']}: {item['trace_id']}")
else:
    print("Logging skipped. Set LOG_TRACES = True to publish the embedded traces.")


Loaded 5 embedded trace(s).
Logged 5 trace(s) to 'multi-agent-demo'.
- baseline: 019f7dcf-a33a-7011-8e65-260f2b7f36f3
- variant_A_poor_parameter_and_plan: 019f7dcf-a46c-7d30-9fa3-6a41900595ba
- variant_B_poor_trajectory_and_tools: 019f7dcf-a4da-7a81-b7c6-098bb26af9f1
- variant_C_poor_summary_accuracy: 019f7dcf-a558-7d01-aa95-2ac0ca689a73
- variant_D_poor_goal_accomplishment: 019f7dcf-a62f-70f1-9db1-af4b617664a8


## 4. Native Metric Prompts (ported verbatim from prompts.py)

Loads the exact registry prompts for the six metrics: the parameter/plan/trajectory `audit` + `overall_reason` prompts, the summary `claim_segmentation` + `claim_verification` prompts, the tool-call system prompt, and the goal-accomplishment builder. These are copied unchanged from `prompts.py`.

In [ ]:
import json

NATIVE_PROMPTS = {
    "parameter_extraction": {
        "audit": '\nRole:\nYou are a STRICT Parameter Extraction Evaluation Engine.\nYou behave deterministically and mechanically.\n \nGoal:\nEvaluate extracted parameters using TWO SEQUENTIAL CHECKS.\n \nInputs:\nINPUT_TEXT:\n{INPUT_TEXT}\n \nEXTRACTED_PARAMETERS:\n{EXTRACTED_PARAMETERS}\n \nEXPECTED_PARAMETER_SCHEMA:\n{EXPECTED_PARAMETER_SCHEMA}\n \n------------------------------------------------------------\nEVALUATION FRAMEWORK (STRICT GATING LOGIC)\n------------------------------------------------------------\n \nYou MUST evaluate parameters using EXACTLY TWO checks:\n \n1. value_check\n2. format_check\n \nAdditionally you MUST output:\n- parameter_name\n- expected_value  (value grounded from INPUT_TEXT)\n- actual_value    (value from EXTRACTED_PARAMETERS)\n \n------------------------------------------------------------\nCHECK DEFINITIONS\n------------------------------------------------------------\n \nCHECK 1 — value_check\n \nFirst, extract the EXPECTED VALUE for each parameter directly from INPUT_TEXT.\n \nexpected_value MUST:\n- Be explicitly grounded in INPUT_TEXT.\n- Never be inferred.\n- Never be null.\n- Reflect normalized enum mapping if applicable.\n \nNORMALIZATION RULE (HIGH PRIORITY):\n \nIf EXTRACTED_PARAMETERS contains normalized classifications,\nthey are valid IF they are grounded in INPUT_TEXT.\n \nExamples:\n"normal goods" → Normal\n"frozen fish" → Frozen\n"glassware" → Fragile\n"fresh apples" → Food\n \nNormalization DOES NOT count as hallucination.\n \nvalue_check = Pass\n- If actual_value matches expected_value semantically.\n \nvalue_check = Fail\n- If actual_value contradicts expected_value.\n- If actual_value is not grounded.\n- If actual_value is fabricated.\n \nCRITICAL GATING RULE:\n \nIf value_check = Fail:\n- format_check MUST be "Fail"\n \n------------------------------------------------------------\n \nCHECK 2 — format_check\n \nEvaluate ONLY if value_check = Pass.\n \nUse EXPECTED_PARAMETER_SCHEMA for type validation.\n \nPass:\n- actual_value matches schema type.\n- If schema type contains "|", treat it as a union — Pass if actual_value matches ANY listed type.\n  Example: "number|string" means both 12 and "12kg" are valid format-wise.\n \nFail:\n- Type mismatch with ALL listed types.\n- Malformed structure.\n \nValid types:\n- string\n- number\n \n------------------------------------------------------------\nSTRICT RULES\n------------------------------------------------------------\n \n- Do NOT infer missing values.\n- Do NOT use external knowledge.\n- Respect gating rules EXACTLY.\n- Every schema parameter MUST appear once in output.\n- parameter_name MUST match schema key exactly.\n- expected_value MUST come from INPUT_TEXT.\n- actual_value MUST come from EXTRACTED_PARAMETERS.\n- No extra commentary outside JSON.\n \n------------------------------------------------------------\nOUTPUT FORMAT (STRICT JSON ARRAY)\n------------------------------------------------------------\n \n[\n  {\n    "parameter_name": "string",\n    "expected_parameter_value": "string",\n    "actual_parameter_value": "string",\n    "value_check": "Pass | Fail",\n    "format_check": "Pass | Fail",\n    "reason": "short deterministic explanation"\n  }\n]\n \n------------------------------------------------------------\nFEW SHOT EXAMPLE\n------------------------------------------------------------\n \nEXPECTED_PARAMETER_SCHEMA:\n{"weight":"number|string"}\n \nINPUT_TEXT:\n"Ship 12kg box"\n \nEXTRACTED_PARAMETERS:\n{"weight":"12kg"}\n \nOutput:\n[\n  {\n    "parameter_name":"weight",\n    "expected_parameter_value":"12kg",\n    "actual_parameter_value":"12kg",\n    "value_check":"Pass",\n    "format_check":"Pass",\n    "reason":"Value grounded and matches union type number|string."\n  }\n]\n',
    },
    "summary_accuracy": {
        "claim_segmentation": '\nRole:\nYou are a CLAIM SEGMENTATION ENGINE.\n\nYour task is to extract ATOMIC FACTUAL OUTPUT FRAGMENTS from a summary.\n\nInputs:\nsummary:\n{SUMMARY}\n\nInstructions:\nDefinition:\nAn Output Fragment is a sentence or clause that asserts something factual\nthat could be verified against execution context.\n\nImportant Rules (STRICT):\n- Do NOT infer information.\n- Do NOT rewrite or paraphrase.\n- Preserve the ORIGINAL wording exactly.\n- Exclude questions, suggestions, and meta statements.\n- Exclude sentences that are purely evaluative, emotional, or conversational.\n- DO NOT verify correctness.\n- DO NOT use external knowledge.\n\n------------------------------------------------------------\nCRITICAL SEGMENTATION RULES (NEW — HIGH PRIORITY)\n------------------------------------------------------------\n\nYou MUST preserve semantic dependency.\n\nDO NOT split fragments when:\n\n1) Parenthetical qualifiers exist:\n   Example:\n   "Estimated cost is $100 (normalized cost matches this amount)"\n   MUST remain ONE fragment.\n\n2) Referential phrases exist:\n   "this amount", "that value", "these results", "which indicates"\n   MUST remain attached to the main clause.\n\n3) Dependent explanatory clauses exist:\n   - normalized\n   - adjusted\n   - derived\n   - calculated\n   - mapped\n   - based on\n\n4) If removing part of the sentence causes ambiguity,\n   the fragment MUST remain whole.\n\nSplit ONLY when:\n- Two independent factual assertions exist with no dependency.\n\n------------------------------------------------------------\nDecision Boundary Rules:\n------------------------------------------------------------\n\nA fragment MUST be trace-verifiable.\n\nDo NOT isolate:\n- Parentheses (...)\n- trailing qualifiers\n- explanatory commas\n- subordinate clauses beginning with "which", "where", "that"\n\n------------------------------------------------------------\nOutput Format:\nReturn STRICT JSON only:\n\n{\n  "claims": ["string"]\n}\n\nFew Shot Examples:\n\nExample 1:\nInput Summary:\n"The plan ID is PLAN_1. It is a FULL delivery."\nOutput:\n{\n  "claims": [\n    "The plan ID is PLAN_1",\n    "It is a FULL delivery"\n  ]\n}\n\nExample 2 - Parenthesis MUST NOT split:\nInput:\n"Estimated cost for the move is $1,264,076 (normalized cost matches this amount)"\nOutput:\n{\n  "claims": [\n    "Estimated cost for the move is $1,264,076 (normalized cost matches this amount)"\n  ]\n}\n\nExample 3:\nInput Summary:\n"Weather conditions are sunny with a risk of 0.1, which should not impact shipment."\nOutput:\n{\n  "claims": [\n    "Weather conditions are sunny with a risk of 0.1"\n  ]\n}\n',
        "claim_verification": '\nYou are a FACTUAL CONSISTENCY VERIFIER.\n\nYour task is to determine whether the given SUMMARY OUTPUT FRAGMENT is SUPPORTED by the provided CONTEXT.\n\nDEFINITIONS:\n- A SUMMARY OUTPUT FRAGMENT is a factual statement extracted from the system output.\n- The CONTEXT is the ONLY source of truth.\n\nIMPORTANT RULES:\n1. Use ONLY CONTEXT.\n2. A fragment is SUPPORTED ONLY IF explicit evidence exists.\n3. Supporting evidence MUST be copied verbatim or near-verbatim.\n4. If no explicit evidence exists -> "No Match".\n5. Do NOT infer or reason beyond text.\n\nContext Search Procedure (MANDATORY):\nYou MUST actively search the CONTEXT to locate supporting evidence.\nThe CONTEXT may contain nested structured data (JSON-like objects).\nYou MUST inspect fields, keys, and values inside the structure.\n\nStep-by-step search strategy:\n1. Identify the subject of the fragment (plan, vendor, route, date, cost, etc.).\n2. Scan the CONTEXT for fields whose key names or values relate to that subject.\n3. Verify whether an explicit value matches the fragment exactly.\n4. You MUST always extract RELEVANT CONTEXT SIGNALS when they exist.\n\nRELEVANT CONTEXT SIGNALS include:\n- matching entities\n- matching attributes\n- matching numeric values\n- related fields that partially align with the fragment\n\nIMPORTANT:\nSupporting Context Evidence is NOT limited to full matches.\nEven when the final status is "No Match", you MUST still include\nany relevant context snippets that relate to the fragment.\n\n5. Evidence Match Status logic:\n\nMatch:\n- explicit statement exists confirming the fragment.\n\nNo Match:\n- context contains related facts but lacks explicit confirmation,\n  OR introduces conflicting semantics.\n\nEvidence presence does NOT imply Match.\n\n6. Do NOT assume relationships between fields unless explicitly stated.\n\nEvidence Selection Rules:\n- Prefer exact key-value matches over descriptive text.\n- Supporting statements must come directly from CONTEXT content.\n- Do NOT summarize or paraphrase the evidence.\n\nExplicit Grounding Rules:\n\n- A JSON field is valid support ONLY if:\n  key semantically matches fragment subject AND\n  value matches fragment attribute.\n\n- Date relationships MUST be explicit.\n  "delivery by X" does NOT imply "requested on X".\n\n- NUMERIC MATCHING RULE (UPDATED - STRICT)\n\nYou MAY treat values as equal when differences are caused ONLY by\nformatting or precision normalization.\n\nAllowed equivalence cases:\n\n1) Formatting normalization:\n   "$1,264,076" == 1264076\n\n2) Float vs integer:\n   1264076 == 1264076.0\n\n3) Precision truncation caused by presentation:\n   98888.01 == 98888\n   ONLY when:\n   - the integer portion is identical\n   - difference is <= 1.0\n   - fragment wording suggests rounding or normalization\n\nSTRICTLY FORBIDDEN:\n- Rounding to nearest thousand/hundred.\n- Percentage or arithmetic reasoning.\n- Any comparison where integer portion differs.\n\nOutput Format:\n{\n  "Supporting Context Evidence": "string",\n  "Evidence Match Status": "Match | No Match",\n  "Verification Reason": "string"\n}\n\nFew Shot Examples:\n\nExample 1 - Direct Match\nInput Fragment:\n"The plan ID is PLAN_1"\nContext:\nPLAN_ID = PLAN_1\nOutput:\n{\n  "Supporting Context Evidence": "PLAN_ID = PLAN_1",\n  "Evidence Match Status": "Match",\n  "Verification Reason": "Context explicitly states PLAN_1."\n}\n\nExample 2 - No Match\nFragment:\n"You requested the shipment on 2026-10-01"\nContext:\ndelivery_date = 2026-10-01\nOutput:\n{\n  "Supporting Context Evidence": "",\n  "Evidence Match Status": "No Match",\n  "Verification Reason": "Delivery date is not equivalent to request date."\n}\n\nExample 3 - Structured JSON Match\nFragment:\n"it is a FULL delivery"\nContext:\n{\'delivery_type\': \'full\'}\nOutput:\n{\n  "Supporting Context Evidence": "\'delivery_type\': \'full\'",\n  "Evidence Match Status": "Match",\n  "Verification Reason": "Explicit structured field confirms delivery type."\n}\n',
    },
    "trajectory_match": {
        "audit": '\nRole:\nYou are an Expert Agent-Trajectory Evaluator.\n\nYour task is to determine whether the ACTUAL agent execution trajectory\nconforms to the REFERENCE agent trajectory under a specific matching MODE.\n\nTradeoff Scenarios (IMPORTANT):\nThe following scenarios define acceptable deviations that MUST be treated as\nneutral behaviour during evaluation if encountered:\n{TRADEOFF_TEXT}\n\nYou must behave deterministically.\nYou must not infer intent, repair execution, or reinterpret system behavior.\n\nInputs:\nreference_outputs:\n{REFERENCE_OUTPUTS_PRETTY}\n\nactual_outputs:\n{ACTUAL_OUTPUTS_PRETTY}\n\nmode:\n{MODE}\n\nInstructions:\n\nMeta Instructions:\nReturn ONLY valid JSON matching output_structure_mandate.\nDo NOT include explanations outside the JSON object.\nDo NOT infer missing agents, intent, or system behavior.\n\nGlobal Context:\nThis evaluation is part of a MULTI-MODE agent trajectory analysis.\nThe same reference_outputs and actual_outputs may be evaluated across multiple modes.\nYour per-mode decision MUST remain logically consistent across modes.\n\nCRITICAL MODE EXCLUSIVITY RULES (MANDATORY):\n\n- If strict mode = Pass → ALL other modes MUST return Fail.\n- subset mode requires: len(actual_outputs) < len(reference_outputs)\n- superset mode requires: len(actual_outputs) > len(reference_outputs)\n- unordered mode requires:\n  same agent names BUT execution order must differ.\n- Modes must NEVER overlap.\n\nDefinitions:\n\nAgent Invocation:\nEvery occurrence of an agent in the sequence is significant.\nDuplicate executions MUST be preserved and compared.\nDo NOT collapse repeated agents into a single occurrence.\n\nReference Outputs:\nThe authoritative, expected agent sequence.\n\nActual Outputs:\nThe observed agent invocation sequence from execution.\n\nMode Semantics:\n\nstrict:\nactual_outputs MUST exactly match reference_outputs.\n\nunordered:\nSame agents but order different.\n\nsubset:\nActual must be smaller than expected.\n\nsuperset:\nActual must be larger than expected.\n\nEvaluation Rules:\n\nThe reason MUST describe:\n- where order differs\n- which agents are missing\n- which agents appear extra\n- how many times agents appear when relevant\n\nThe reason MUST include a short comparison narrative like:\n"reference has X while actual has Y".\n\nUse simple wording:\nUse "larger", "smaller", "extra", "missing".\nAvoid complex phrases.\n\nAnalysis Approach:\n\n1. Compare sequences.\n2. Check ordering.\n3. Count extra occurrences.\n4. Count missing occurrences.\n5. Produce a detailed but simple explanation.\n\nOutput Format:\n\n{\n  "match": "Pass | Fail",\n  "mode": "{MODE}",\n  "reference_sequence": {REFERENCE_OUTPUTS_INLINE},\n  "actual_sequence": {ACTUAL_OUTPUTS_INLINE},\n  "missing_agents": ["string"],\n  "extra_agents": ["string"],\n  "reason": "Detailed comparison explanation"\n}\n',
        "overall_reason": '\nReturn STRICT JSON only.\n\nRole:\nYou are an evaluation summarizer.\n\nUse SIMPLE wording.\n\nExplain overall trajectory behaviour using all modes.\n\nInputs:\n{PER_MODE_SUMMARY}\n\nOutput:\n{\n  "overall_reason": "string"\n}\n',
    },
    "plan_accuracy": {
        "audit": '\nYou are a STRICT CONTEXT-AWARE PLAN STRUCTURE AUDITOR.\n\nYour responsibility is to audit a GENERATED EXECUTION PLAN\nagainst SUPERVISOR WORKFLOW RULES and STRUCTURAL FLOW LOGIC.\n\nYou are NOT evaluating writing quality, grammar, or style.\nYou are performing a STRUCTURAL WORKFLOW AUDIT.\n\nYou MUST behave deterministically and mechanically.\n\n============================================================\nPRIMARY OBJECTIVE\n============================================================\n\nYou MUST perform ALL of the following:\n\n1) SEMANTIC STEP FRAGMENTATION\n2) RULE TEXT MATCHING\n3) RULE ACCURACY VALIDATION\n4) FLOW ACCURACY VALIDATION\n\nThese MUST be done in ONE PASS.\n\n============================================================\nTASK 1 — SEMANTIC STEP FRAGMENTATION (STRICT)\n============================================================\n\nYou MUST extract MEANINGFUL EXECUTION STEPS from generated_plan.\n\nA VALID fragment MUST:\n\n✔ represent a COMPLETE execution action\n✔ contain an AGENT or EXECUTION SUBJECT\n✔ be independently auditable\n\nDO NOT split sentences based on grammar alone.\n\nVALID EXAMPLES:\n- "Step 1: Invoke ContextBuilder with parameters..."\n- "TaskPlanner generates routing combinations"\n\nINVALID FRAGMENTS:\n- "generates routing combinations"\n- "and then sends email"\n- clauses without agent/action\n\n------------------------------------------------------------\nFRAGMENTATION SIGNALS (HIGH PRIORITY)\n------------------------------------------------------------\n\nPrefer splitting using:\n\n- "Step X:"\n- "Task X:"\n- Explicit agent transitions\n- Workflow stage boundaries\n\n============================================================\nTASK 2 — CONTEXT PRESERVATION (CRITICAL)\n============================================================\n\nYou MUST internally carry PREVIOUS steps as background context\nwhen evaluating a fragment.\n\nExample:\n\nStep 1: ContextBuilder gathers data\nStep 2: Forecast computes costs\n\nWhen evaluating Step 2:\n\n✔ consider Step 1 internally\n❌ DO NOT include Step 1 text in Output Fragment\n\nOutput Fragment MUST contain ONLY the CURRENT STEP TEXT.\n\n============================================================\nTASK 3 — RULE MATCHING (STRICT TEXT MATCH)\n============================================================\n\nFor EACH Output Fragment:\n\nYou MUST locate the EXACT matching rule text inside supervisor_rules.\n\nRules:\n\n✔ Copy rule text VERBATIM\n✔ Do NOT paraphrase\n✔ Do NOT synthesize new rules\n✔ Do NOT merge multiple rules\n\nIf NO matching rule exists:\n\nReturn:\n"Matched Rule Fragment": "N/A"\n\n------------------------------------------------------------\nRULE MATCHING STRATEGY\n------------------------------------------------------------\n\nMatch based on:\n\n- Agent name\n- Execution responsibility\n- Action semantics\n\nExample:\n\nFragment:\n"Optimization ranks the plans"\n\nMatching rule:\n"6. **Optimization**: Ranks the plans based on utility..."\n\n============================================================\nTASK 4 — PLAN OUTPUT RULE ACCURACY\n============================================================\n\npass:\n✔ Agent exists in supervisor_rules\n✔ Action allowed by rules\n✔ No forbidden behaviour\n\nfail:\n✖ Agent not present in rules\n✖ Action contradicts rule text\n✖ Required workflow stage skipped\n\nIMPORTANT:\nRule Accuracy MUST be judged ONLY using supervisor_rules.\n\n============================================================\nTASK 5 — PLAN OUTPUT FLOW ACCURACY\n============================================================\n\npass:\n✔ Logical ordering preserved\n✔ Dependencies satisfied\n✔ Workflow progression valid\n\nfail:\n✖ Execution before prerequisite\n✖ Circular logic\n✖ Impossible ordering\n\nFlow Accuracy evaluates INTERNAL PLAN STRUCTURE ONLY.\n\n============================================================\nSTRICT GLOBAL CONSTRAINTS\n============================================================\n\n- DO NOT rewrite fragments\n- DO NOT invent rules\n- DO NOT expose hidden context\n- DO NOT evaluate language quality\n- DO NOT combine fragments\n- DO NOT reference user intent in scoring\n- ALWAYS return JSON only\n\n============================================================\nINPUTS\n============================================================\n\nuser_input:\n{USER_INPUT}\n\nsupervisor_rules:\n{RULES}\n\ngenerated_plan:\n{PLAN}\n\n============================================================\nOUTPUT FORMAT (STRICT JSON)\n============================================================\n\n{\n  "fragments":[\n    {\n      "Output Fragment":"string",\n      "Matched Rule Fragment":"string | N/A",\n      "Plan Output Rule Accuracy":"pass | fail",\n      "Plan Output Flow Accuracy":"pass | fail",\n      "Rule Accuracy Reason":"string",\n      "Flow Accuracy Reason":"string"\n    }\n  ]\n}\n\n============================================================\nFEW SHOT EXAMPLES\n============================================================\n\nExample 1 — Perfect Alignment\n\nSupervisor Rule:\n"3. **TaskPlanner**: Formulates shipment order..."\n\nFragment:\n"Step 3: TaskPlanner formulates shipment order..."\n\nOutput:\n{\n  "Output Fragment":"Step 3: TaskPlanner formulates shipment order...",\n  "Matched Rule Fragment":"3. **TaskPlanner**: Formulates shipment order...",\n  "Plan Output Rule Accuracy":"pass",\n  "Plan Output Flow Accuracy":"pass",\n  "Rule Accuracy Reason":"Fragment directly matches TaskPlanner rule.",\n  "Flow Accuracy Reason":"Correctly follows prior verification stage."\n}\n\n------------------------------------------------------------\n\nExample 2 — Rule Violation\n\nSupervisor Rule:\n"Only Optimization ranks vendors."\n\nFragment:\n"Forecast ranks vendors by score"\n\nOutput:\n{\n  "Output Fragment":"Forecast ranks vendors by score",\n  "Matched Rule Fragment":"N/A",\n  "Plan Output Rule Accuracy":"fail",\n  "Plan Output Flow Accuracy":"pass",\n  "Rule Accuracy Reason":"Forecast is not authorized to rank vendors.",\n  "Flow Accuracy Reason":"Ordering itself is logical."\n}\n\n------------------------------------------------------------\n\nExample 3 — Flow Violation\n\nFragment:\n"Execute sends email before Compliance checks"\n\nOutput:\n{\n  "Output Fragment":"Execute sends email before Compliance checks",\n  "Matched Rule Fragment":"10. **Execute**: Saves the confirmed order to DB and sends an email.",\n  "Plan Output Rule Accuracy":"pass",\n  "Plan Output Flow Accuracy":"fail",\n  "Rule Accuracy Reason":"Execute action allowed by rules.",\n  "Flow Accuracy Reason":"Execution occurs before compliance which violates workflow ordering."\n}\n',
        "overall_reason": '\nYou are a PLAN ACCURACY SUMMARIZER.\n\nExplain structural correctness patterns.\n\nUse SIMPLE wording.\nDescribe ONLY:\n- rule accuracy trends\n- flow accuracy trends\n\nInput:\n{FRAGMENT_RESULTS}\n\nReturn a JSON object with this structure:\n{"overall_reason": "your summary text here"}\n',
    },
}

TOOL_CALL_ACCURACY_SYSTEM_PROMPT = '\n    Role:\nYou are an AUTOMATED TOOL CALL VERIFICATION ENGINE.\nYou are a STRICT, MECHANICAL AUDITOR.\nYou are NOT a planner, assistant, or problem solver.\nYour ONLY responsibility is to verify whether tool calls that were ACTUALLY EXECUTED conform exactly to their authoritative specifications.\nYou must behave deterministically.\nYou must not infer, repair, or reinterpret behavior.\n\nInputs:\n{EXPECTED_TOOLS}: Authoritative specification of valid tools. Defines valid tool names, allowed parameter names per tool, parameter_policy ("exact" or "any_subset"), whether parameter format checking is enforced via "strict", and parameter formats (if defined).\n{ACTUAL_TOOL_CALLS}: The complete set of tool calls that were executed. Defines the ONLY evaluation universe.\n\nInstructions:\nEvaluation Scope (Critical):\nEvaluate ONLY tools listed in ACTUAL_TOOL_CALLS.\nDo NOT invent, infer, or mention tools that were not called.\nDo NOT penalize missing tool calls.\nDo NOT apply external or domain knowledge.\n\nEvaluation Order (Strict and Mandatory):\nYou MUST apply checks in the following order, with NO deviation:\n1. Tool Name Check\n2. Tool Parameter Check\n3. Parameter Format Check\n\nGating Rules (Mandatory Control Flow):\nIf Tool Name Check = incorrect:\n- Tool Parameter Check → skipped\n- Parameter Format Check → skipped\n\nIf Tool Parameter Check = incorrect:\n- Parameter Format Check → skipped\n\nCheck Definitions:\n\n1. Tool Name Check\ncorrect:\n- tool_name exists as a key in EXPECTED_TOOLS\nincorrect:\n- tool_name does NOT exist in EXPECTED_TOOLS\n\n2. Tool Parameter Check\nValidation depends on parameter_policy defined in EXPECTED_TOOLS.\n\nparameter_policy = "exact":\n- Provided parameter keys MUST match EXACTLY\n- No missing parameters\n- No extra parameters\n\nparameter_policy = "any_subset":\n- Provided parameter keys MUST be a subset of allowed_parameters\n- Any extra parameter NOT listed is INVALID\n\nIf Tool Name Check is incorrect, this check MUST be skipped.\n\n3. Parameter Format Check\nValidate ONLY parameter VALUE formats.\nParameter formats are defined INSIDE EXPECTED_TOOLS.\nDo NOT infer or assume formats.\nIf no format is defined for a parameter, do NOT validate it.\n\nIf Tool Parameter Check is incorrect, this check MUST be skipped.\n\nAllowed Judgment Values:\nYou MUST use ONLY:\n"pass"\n"fail"\n\nRules:\n- Any skipped check MUST be marked as "fail".\n- Do NOT output "skipped".\n\n\nReason Construction Rules (STRICT):\n\nThe reason field MUST include the expected schema formats\nfor ALL provided arguments.\n\nFormat requirements:\n\n- Use the word "arguments" (never parameters).\n- When argument format check passes, append the expected formats\n  in brackets using this pattern:\n\n  expected_formats: arg1(type), arg2(type)\n\nExample:\n"Tool recognized; provided arguments comply with exact policy; argument value types align with schema expectations; expected_formats: vendor_id_1(string), vendor_id_2(string)."\n\nRules:\n\n- expected_formats MUST come from EXPECTED_TOOLS.allowed_parameters.\n- Include ONLY arguments that were provided in the ACTUAL_TOOL_CALL.\n- Maintain professional audit tone.\n- Do NOT add recommendations or suggestions.\n\n\nOutput Format:\nReturn ONLY a JSON object with the following structure:\n{\n  "tools": [\n    {\n      "tool_name": "string",\n      "tool_name_check": "pass | fail",\n      "tool_parameter_check": "pass | fail",\n      "parameter_format_check": "pass | fail",\n      "reason": "short, precise justification"\n    }\n  ],\n  "overall_reason": "High-level assessment of tool-call correctness"\n}\n\nFew Shot Examples:\n\nExample 1 — Fully Correct Tool Call\n\nEXPECTED_TOOLS:\n{\n  "analytics_vendors_find": {\n    "allowed_parameters": ["base_location", "limit"],\n    "parameter_policy": "any_subset",\n    "strict": false\n  }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_vendors_find",\n    "parameters": {\n      "base_location": "BOSTON"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_vendors_find",\n      "tool_name_check": "pass",\n      "tool_parameter_check": "pass",\n      "parameter_format_check": "pass",\n      "reason": "Tool recognized; provided arguments comply with subset policy; argument value types align with schema expectations; expected_formats: base_location(string)."\n    }\n  ],\n  "overall_reason": "All executed tools conform to their specifications."\n}\n\nExample 2 — Incorrect Tool Name\n\nEXPECTED_TOOLS:\n{\n  "analytics_vendors_find": { }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_vendor_search",\n    "parameters": {\n      "base_location": "BOSTON"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_vendor_search",\n      "tool_name_check": "fail",\n      "tool_parameter_check": "fail",\n      "parameter_format_check": "fail",\n      "reason": "Tool not recognized; executed tool is not present in EXPECTED_TOOLS, therefore argument validation and format evaluation cannot be performed."\n    }\n  ],\n  "overall_reason": "At least one executed tool is not defined in EXPECTED_TOOLS."\n}\n\nExample 3 — Argument Mismatch (Exact Policy)\n\nEXPECTED_TOOLS:\n{\n  "analytics_confirmed_order_lookup": {\n    "allowed_parameters": ["order_id"],\n    "parameter_policy": "exact",\n    "strict": false\n  }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_confirmed_order_lookup",\n    "parameters": {\n      "order_id": "123",\n      "status": "confirmed"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_confirmed_order_lookup",\n      "tool_name_check": "pass",\n      "tool_parameter_check": "fail",\n      "parameter_format_check": "fail",\n      "reason": "Tool recognized; provided arguments violate exact policy due to unexpected argument \'status\'; expected_formats: order_id(string)."\n    }\n  ],\n  "overall_reason": "One or more executed tools violate argument policy constraints."\n}\n    '


# ---- Builder functions copied verbatim from prompts.py ----
def build_parameter_extraction_overall_reason_prompt(audit_results):
    return f'''
Role:
You are a STRICT Evaluation Summary Generator.

Purpose:
Produce a concise structural summary describing grounding behaviour.

CRITICAL:
- Do NOT provide suggestions.
- Do NOT mention format/type issues.
- Describe only grounding (value_check).
- Return ONLY raw JSON. No markdown, no code fences.

audit_results:
{json.dumps(audit_results, indent=2)}

Return ONLY this JSON object (no markdown, no extra text):

{{
  "overall_reason": "1-3 concise factual sentences."
}}
'''


def build_tool_call_accuracy_user_prompt(expected_tools, slimmed_tool_calls):
    return f"""
EXPECTED TOOLS:
{json.dumps(expected_tools, indent=2)}

ACTUAL TOOL CALLS:
{json.dumps(slimmed_tool_calls, indent=2)}
"""


def build_goal_accomplishment_prompt(task, actual_outcome):
    return f'''meta_instruction:
Produce ONLY the required JSON object, with no commentary outside the JSON.

metric_name:
Goal Accomplishment

role:
You are a STRICT Goal Accomplishment Evaluator. You determine whether the actual_outcome
DIRECTLY and CONCRETELY fulfills each explicit requirement in the user's task.

inputs:

task:
{task}

actual_outcome:
{actual_outcome}

definitions:

User Query Fragment:
A concrete requirement or intent in the task (an analysis to deliver, a decision to make,
a recommendation to produce, or a constraint to satisfy).

Match (STRICT):
- The actual_outcome contains a CONCRETE, SPECIFIC statement that fulfills the fragment
  (for example a named recommendation, specific markets, segments, actions, or timing,
  or actual findings).

No Match:
- The fragment is absent, OR
- The actual_outcome only acknowledges the topic, defers it, or is vague / non-committal
  (for example "it is important", "keep researching", "consider ... at some point",
  "expand when the timing seems right"). Deferring or merely restating the task is NOT
  accomplishment.

analysis_approach:
1. Break the task into explicit query fragments grounded only in the task text.
2. For each fragment, search actual_outcome for a concrete fulfilling statement.
3. Mark Match only if such a concrete statement exists; otherwise No Match.

scoring:
verdict = (number of Match fragments) / (total fragments), rounded to 2 decimals.

strict_rules:
- Evaluate ONLY the provided inputs. Do NOT infer hidden intent.
- Do NOT reward intentions, plans to act later, or generic statements of importance.
- IGNORE process metadata such as approval flags, iteration counts, or which agents ran;
  judge ONLY the substantive answer to the task.
- Judge accomplishment, not writing quality or style.

output_structure_mandate:

Return ONLY JSON:

{{
  "verdict": number,
  "reason": "1-3 sentences summarizing overall goal accomplishment",
  "fragments": [
    {{
      "query_fragment": "string",
      "summary_evidence": "string",
      "status": "Match | No Match"
    }}
  ]
}}

few_shot_examples:

Example 1 (concrete -> Match):
task:
Ship frozen salmon by air and deliver before Feb 5.
actual_outcome:
Plan uses air freight via UPS; delivery ETA Feb 4.
Output:
{{
  "verdict": 1.0,
  "reason": "All requirements are concretely met.",
  "fragments": [
    {{"query_fragment": "air shipment", "summary_evidence": "air freight via UPS", "status": "Match"}},
    {{"query_fragment": "deliver before Feb 5", "summary_evidence": "ETA Feb 4", "status": "Match"}}
  ]
}}

Example 2 (vague / deferring -> No Match):
task:
Analyze the electric vehicle market in Europe and recommend a market-entry strategy for 2027.
actual_outcome:
Europe is an important electric vehicle market; the company should keep researching and consider expanding at some point when the timing is right.
Output:
{{
  "verdict": 0.0,
  "reason": "The outcome only calls the market important and defers the decision; it gives no competitive analysis and no concrete 2027 entry strategy.",
  "fragments": [
    {{"query_fragment": "analyze the competitive landscape for electric vehicles in Europe", "summary_evidence": "only says Europe is important", "status": "No Match"}},
    {{"query_fragment": "recommend a market-entry strategy for 2027", "summary_evidence": "defers: keep researching / consider later", "status": "No Match"}}
  ]
}}
'''.strip()


print(f"Loaded native prompts for {len(NATIVE_PROMPTS)} registry metrics "
      f"+ tool_call system prompt + 3 builder functions.")


Loaded native prompts for 4 registry metrics + tool_call system prompt + 3 builder functions.


## 5. Per-Trace Expected Config (Excel)

The expected trajectory and expected tool schema are no longer hard-coded. They live in `expected_config.xlsx` — one row per trace, keyed by the `scenario` value carried in each root trace's input. This cell creates the workbook on first run (seeded with the ideal workflow) and loads it; edit any row to give a trace its own unique expected trajectory/tools, then re-run. `resolve_expected_config()` maps each trace to its row by reading the root input.


In [ ]:
# ============================================================
# Per-Trace Expected Config (Excel-driven)
# ============================================================
# The expected trajectory and expected tool schema are no longer hard-coded.
# They live in an Excel workbook, ONE ROW PER TRACE, keyed by the "scenario"
# value found in each root trace's INPUT. At evaluation time each trace is
# mapped to its expected row by reading that input key.

import json
from pathlib import Path

EXPECTED_CONFIG_FILE = "/content/expected_config.xlsx"
TRAJECTORY_DELIMITER = " | "

# ---- Seed content used ONLY to generate the workbook the first time. ----
# Once the file exists it is the source of truth: edit the Excel (per row) to
# give any trace its own unique expected trajectory / tools, then re-run.
_CANONICAL_TRAJECTORY = [
    "Parameter Extraction Agent", "Supervisor: Plan & Route",
    "Research Agent", "Web Search Sub-Agent", "Document Retrieval Sub-Agent",
    "Source Validator Sub-Agent", "Supervisor: Review Research",
    "Analysis Agent", "Market Sizing Sub-Agent", "Competitor Benchmarking Sub-Agent",
    "Supervisor: Review Analysis", "Research Agent", "Web Search Sub-Agent",
    "Supervisor: Review Addendum", "Report Writer Agent", "Drafting Sub-Agent",
    "Formatting & Citation Sub-Agent", "Supervisor: Final Review",
]
_CANONICAL_TOOLS = {
    "Web Search Sub-Agent": {"allowed_parameters": ["tool_name", "search_query", "purpose"], "parameter_policy": "any_subset", "strict": False},
    "Document Retrieval Sub-Agent": {"allowed_parameters": ["search_query", "document_type", "purpose"], "parameter_policy": "any_subset", "strict": False},
    "Source Validator Sub-Agent": {"allowed_parameters": ["claims_to_verify", "verification_method"], "parameter_policy": "any_subset", "strict": False},
    "Market Sizing Sub-Agent": {"allowed_parameters": ["tool_name", "parameters"], "parameter_policy": "any_subset", "strict": False},
    "Competitor Benchmarking Sub-Agent": {"allowed_parameters": ["competitors_to_compare", "comparison_dimensions"], "parameter_policy": "any_subset", "strict": False},
    "Drafting Sub-Agent": {"allowed_parameters": ["messages"], "parameter_policy": "any_subset", "strict": False},
    "Formatting & Citation Sub-Agent": {"allowed_parameters": ["draft_text", "sources_to_cite"], "parameter_policy": "any_subset", "strict": False},
}
# One row per trace; keys MUST match the "scenario" set on each root input.
_SEED_KEYS = [
    "baseline",
    "variant_A_poor_parameter_and_plan",
    "variant_B_poor_trajectory_and_tools",
    "variant_C_poor_summary_accuracy",
    "variant_D_poor_goal_accomplishment",
]


def _seed_expected_config_excel(path):
    rows = [{
        "key": key,
        "expected_trajectory": TRAJECTORY_DELIMITER.join(_CANONICAL_TRAJECTORY),
        "expected_tools": json.dumps(_CANONICAL_TOOLS),
    } for key in _SEED_KEYS]
    pd.DataFrame(rows, columns=["key", "expected_trajectory", "expected_tools"]).to_excel(path, index=False)


if not Path(EXPECTED_CONFIG_FILE).exists():
    _seed_expected_config_excel(EXPECTED_CONFIG_FILE)
    print(f"Created expected-config workbook '{EXPECTED_CONFIG_FILE}' with {len(_SEED_KEYS)} row(s).")
else:
    print(f"Using existing expected-config workbook '{EXPECTED_CONFIG_FILE}'.")


def load_expected_config(path=EXPECTED_CONFIG_FILE):
    """Read the workbook into {key: {"trajectory": [...], "tools": {...}}}."""
    df = pd.read_excel(path, dtype=str).fillna("")
    config = {}
    for _, row in df.iterrows():
        key = str(row["key"]).strip()
        if not key:
            continue
        trajectory = [s.strip() for s in str(row["expected_trajectory"]).split(TRAJECTORY_DELIMITER)
                      if s.strip()]
        try:
            tools = json.loads(row["expected_tools"]) if str(row["expected_tools"]).strip() else {}
        except json.JSONDecodeError:
            tools = {}
        config[key] = {"trajectory": trajectory, "tools": tools}
    return config


EXPECTED_CONFIG_BY_KEY = load_expected_config()


def resolve_expected_config(execution_trace):
    """Map a trace to its expected row via the 'scenario' key in the ROOT input."""
    key = (execution_trace.get("inputs") or {}).get("scenario")
    cfg = EXPECTED_CONFIG_BY_KEY.get(str(key)) if key is not None else None
    if cfg is None:
        # If every row shares one expected config, fall back to it; otherwise empty.
        uniq = {json.dumps(v, sort_keys=True): v for v in EXPECTED_CONFIG_BY_KEY.values()}
        cfg = next(iter(uniq.values())) if len(uniq) == 1 else {"trajectory": [], "tools": {}}
    return cfg


print(f"Loaded expected config for {len(EXPECTED_CONFIG_BY_KEY)} trace key(s): "
      f"{', '.join(EXPECTED_CONFIG_BY_KEY)}")


Using existing expected-config workbook '/content/expected_config.xlsx'.
Loaded expected config for 5 trace key(s): baseline, variant_A_poor_parameter_and_plan, variant_B_poor_trajectory_and_tools, variant_C_poor_summary_accuracy, variant_D_poor_goal_accomplishment


## 6. Selected Metrics

Lists the six metrics executed for every retrieved trace.


In [ ]:
# ============================================================
# Metric Selection
# ============================================================

SELECTED_METRICS = [
    "trajectory_match",
    "parameter_extraction",
    "plan_accuracy",
    "tool_call_accuracy",
    "summary_accuracy",
    "goal_accomplishment",
]

print("Selected evaluation metrics:")
for index, metric in enumerate(SELECTED_METRICS, start=1):
    print(f"{index}. {metric}")


Selected evaluation metrics:
1. trajectory_match
2. parameter_extraction
3. plan_accuracy
4. tool_call_accuracy
5. summary_accuracy
6. goal_accomplishment


## 7. Retrieve and Reconstruct Recent Traces

Set `LOOKBACK_HOURS` to bound the search window, then keep only the **latest run per trace name** (re-running the logger publishes a fresh batch, so older duplicates in the window are dropped). Each selected root trace is reconstructed into its full parent-child hierarchy.


In [ ]:
# ============================================================
# Retrieve LangSmith Traces in a Time Window
# ============================================================

LOOKBACK_HOURS = 1

if LOOKBACK_HOURS <= 0:
    raise ValueError("LOOKBACK_HOURS must be greater than zero.")

window_end = datetime.now(timezone.utc)
window_start = window_end - timedelta(hours=LOOKBACK_HOURS)

root_runs = list(
    langsmith_client.list_runs(
        project_name=LANGSMITH_PROJECT,
        is_root=True,
        start_time=window_start,
    )
)

# Keep only the LATEST run per trace name (re-running the logger publishes a
# fresh batch; older duplicates within the window are dropped), then order by time.
_latest_by_name = {}
for _run in root_runs:
    _st = _run.start_time or window_start
    _prev = _latest_by_name.get(_run.name)
    if _prev is None or (_prev.start_time or window_start) < _st:
        _latest_by_name[_run.name] = _run
root_runs = sorted(_latest_by_name.values(), key=lambda run: run.start_time or window_start)

if not root_runs:
    raise RuntimeError(
        f"No root traces found in project '{LANGSMITH_PROJECT}' "
        f"during the last {LOOKBACK_HOURS} hour(s)."
    )

print("=" * 70)
print("LangSmith Traces Loaded")
print("=" * 70)
print(f"Project          : {LANGSMITH_PROJECT}")
print(f"Lookback window  : last {LOOKBACK_HOURS} hour(s)")
print(f"Window start UTC : {window_start.isoformat()}")
print(f"Window end UTC   : {window_end.isoformat()}")
print(f"Root traces      : {len(root_runs)} (latest run per trace name)")

for index, run in enumerate(root_runs, start=1):
    print(f"{index}. {run.name} | {run.id} | {run.start_time}")

print("=" * 70)


def build_trace_tree(run):
    """Recursively reconstruct one LangSmith execution tree."""
    children = list(langsmith_client.list_runs(parent_run_id=run.id))
    children.sort(key=lambda child: child.start_time or run.start_time)

    return {
        "id": str(run.id),
        "name": run.name,
        "run_type": run.run_type,
        "inputs": run.inputs or {},
        "outputs": run.outputs or {},
        "metadata": run.extra or {},
        "start_time": run.start_time,
        "end_time": run.end_time,
        "children": [build_trace_tree(child) for child in children],
    }


execution_traces = [build_trace_tree(root_run) for root_run in root_runs]

print(f"Reconstructed {len(execution_traces)} trace hierarchy/hierarchies.")


LangSmith Traces Loaded
Project          : multi-agent-demo
Lookback window  : last 1 hour(s)
Window start UTC : 2026-07-20T04:05:50.454215+00:00
Window end UTC   : 2026-07-20T05:05:50.454215+00:00
Root traces      : 5 (latest run per trace name)
1. Supervisor Agent | 019f7dcf-a33a-7011-8e65-260f2b7f36f3 | 2026-07-20 04:36:27.578949+00:00
2. Supervisor Agent — variant_A_poor_parameter_and_plan | 019f7dcf-a46c-7d30-9fa3-6a41900595ba | 2026-07-20 04:36:27.884083+00:00
3. Supervisor Agent — variant_B_poor_trajectory_and_tools | 019f7dcf-a4da-7a81-b7c6-098bb26af9f1 | 2026-07-20 04:36:27.994592+00:00
4. Supervisor Agent — variant_C_poor_summary_accuracy | 019f7dcf-a558-7d01-aa95-2ac0ca689a73 | 2026-07-20 04:36:28.120160+00:00
5. Supervisor Agent — variant_D_poor_goal_accomplishment | 019f7dcf-a62f-70f1-9db1-af4b617664a8 | 2026-07-20 04:36:28.335972+00:00
Reconstructed 5 trace hierarchy/hierarchies.


## 8. Native Multi-Stage Evaluation Working

Reproduces each metric's original working instead of a single judge call: parameter/plan run an audit then an overall-reason summary; trajectory runs the audit once per mode (strict/unordered/subset/superset) then summarizes; summary segments claims then verifies each against the trace; tool-call runs one gated system+user audit; goal-accomplishment runs one fragment-level audit. Every metric keeps its per-fragment output as `traceback` and derives a 1-10 `score` so the downstream report, Excel, and feedback cells are unchanged.

In [ ]:

from collections import defaultdict

PARAMETER_SCHEMA = {
    "company_profile": "object",
    "market": "object",
    "target_year": "number",
    "objective": "string",
    "deliverables": "array",
}

TRAJECTORY_MODES = ["strict", "unordered", "subset", "superset"]


# ------------------------------------------------------------
# LLM call + tolerant JSON parsing
# ------------------------------------------------------------
def invoke_llm(system_prompt, user_prompt, force_json_object=True):
    """Call Azure OpenAI. force_json_object=False for prompts returning a top-level array."""
    kwargs = dict(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    if force_json_object:
        kwargs["response_format"] = {"type": "json_object"}
    return azure_client.chat.completions.create(**kwargs).choices[0].message.content


def parse_json(response):
    """Parse a JSON object OR array, tolerating code fences / surrounding text."""
    txt = (response or "").strip()
    if txt.startswith("```"):
        txt = txt.split("\n", 1)[1] if "\n" in txt else ""
        if txt.rstrip().endswith("```"):
            txt = txt.rstrip()[:-3]
    starts = [i for i in (txt.find("{"), txt.find("[")) if i >= 0]
    if not starts:
        raise ValueError("No JSON object or array found in model response.")
    obj, _ = json.JSONDecoder().raw_decode(txt[min(starts):])
    return obj


def _score_from_ratio(passed, total):
    """Map a pass ratio to a 1-10 integer score (None when nothing to score)."""
    if not total:
        return None
    return max(1, min(10, round(10 * passed / total)))


def _missing(metric, msg):
    return {"metric": metric, "score": None, "max_score": 10,
            "reasoning": msg}


# ------------------------------------------------------------
# Trace preprocessing (unchanged in spirit) + helpers
# ------------------------------------------------------------
def preprocess_execution_trace(execution_trace, expected_tool_names=None):
    expected_tool_names = set(expected_tool_names or ())
    flattened, agent_spans, tool_spans = [], [], []
    span_lookup = defaultdict(list)

    def visit(node):
        flattened.append(node)
        span_lookup[node["name"]].append(node)
        if node["run_type"] in ("tool", "retriever") or node["name"] in expected_tool_names:
            tool_spans.append(node)
        elif node["run_type"] in ("llm", "chain"):
            agent_spans.append(node)
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace)
    return {"flattened_trace": flattened, "agent_spans": agent_spans,
            "tool_spans": tool_spans, "span_lookup": span_lookup}


def first_span(span_lookup, name):
    matches = span_lookup.get(name, [])
    return matches[0] if matches else None


def span_inputs(span):
    return (span or {}).get("inputs", {}) or {}


def span_outputs(span):
    return (span or {}).get("outputs", {}) or {}


def ordered_trajectory_names(execution_trace):
    """Ordered agent-name sequence (root wrapper excluded)."""
    names = []

    def visit(node, is_root=False):
        if not is_root:
            names.append(node["name"])
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace, is_root=True)
    return names


def collect_evidence_context(span):
    """Upstream evidence a summary should be verified against: the span's own
    inputs plus every descendant (sub-agent / tool) output. Excludes the span's
    OWN summary output AND any descendant that merely re-emits that summary
    (e.g. the Drafting / Formatting writers), which would make verification
    circular -- the report would otherwise 'verify' against a copy of itself."""
    own_texts = [v for v in span_outputs(span).values()
                 if isinstance(v, str) and len(v) > 40]
    evidence = {"inputs": span_inputs(span), "evidence": {}}

    def echoes(child):
        blob = json.dumps(span_outputs(child), default=str)
        return any(t in blob for t in own_texts)

    def gather(node):
        for child in node.get("children", []):
            if not echoes(child):
                evidence["evidence"][child["name"]] = span_outputs(child)
            gather(child)

    gather(span)
    return json.dumps(evidence, default=str)


# ------------------------------------------------------------
# Per-metric runners (native working)
# ------------------------------------------------------------
def _detail_suffix(label, items, limit=4, maxlen=120):
    """Fold specifics from the audit into the reasoning string (score unaffected)."""
    items = [str(i).strip() for i in items if str(i).strip()]
    if not items:
        return ""
    shown = items[:limit]
    shown = [s if len(s) <= maxlen else s[:maxlen - 1] + "…" for s in shown]
    tail = f" (+{len(items) - len(shown)} more)" if len(items) > len(shown) else ""
    return f" {label}: " + "; ".join(shown) + tail + "."


def run_parameter_extraction(span):
    if not span:
        return _missing("parameter_extraction", "Parameter Extraction Agent span not found.")
    outputs = span_outputs(span)
    extracted = outputs.get("parameters", outputs)
    audit_prompt = (
        NATIVE_PROMPTS["parameter_extraction"]["audit"]
        .replace("{INPUT_TEXT}", json.dumps(span_inputs(span), indent=2, default=str))
        .replace("{EXTRACTED_PARAMETERS}", json.dumps(extracted, indent=2, default=str))
        .replace("{EXPECTED_PARAMETER_SCHEMA}", json.dumps(PARAMETER_SCHEMA))
    )
    raw = invoke_llm(audit_prompt, "Return the strict JSON array now.", force_json_object=False)
    audit = parse_json(raw)
    if isinstance(audit, dict):
        audit = [audit]
    # Score on value_check (grounding). The native prompt's overall_reason
    # summarizes grounding only, and format_check recognizes just string/number,
    # which cannot validate the nested object/array params in this domain.
    # format_check is computed by the audit but not used for scoring.
    grounded = sum(1 for r in audit if r.get("value_check") == "Pass")
    score = _score_from_ratio(grounded, len(audit))
    reasoning = ""
    try:
        reason_raw = invoke_llm(build_parameter_extraction_overall_reason_prompt(audit),
                                "Return the JSON object now.")
        reasoning = parse_json(reason_raw).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    failed_params = [r.get("parameter_name", "?") for r in audit
                     if r.get("value_check") != "Pass"]
    if failed_params:
        reasoning += _detail_suffix("Parameters not grounded in the request", failed_params, limit=8)
    elif audit:
        reasoning += f" All {len(audit)} extracted parameters are grounded in the request."
    return {"metric": "parameter_extraction", "score": score, "max_score": 10,
            "reasoning": reasoning, "raw_evaluation": raw}


def run_trajectory_match(expected_names, execution_trace):
    actual_names = ordered_trajectory_names(execution_trace)
    ref_pretty, ref_inline = json.dumps(expected_names, indent=2), json.dumps(expected_names)
    act_pretty, act_inline = json.dumps(actual_names, indent=2), json.dumps(actual_names)

    per_mode = []
    for mode in TRAJECTORY_MODES:
        ap = (
            NATIVE_PROMPTS["trajectory_match"]["audit"]
            .replace("{TRADEOFF_TEXT}", "None. No acceptable deviations are defined; evaluate literally.")
            .replace("{REFERENCE_OUTPUTS_PRETTY}", ref_pretty)
            .replace("{ACTUAL_OUTPUTS_PRETTY}", act_pretty)
            .replace("{MODE}", mode)
            .replace("{REFERENCE_OUTPUTS_INLINE}", ref_inline)
            .replace("{ACTUAL_OUTPUTS_INLINE}", act_inline)
        )
        try:
            res = parse_json(invoke_llm(ap, f"Evaluate strictly under mode='{mode}'. Return JSON now."))
        except Exception as e:
            res = {"mode": mode, "match": "Fail", "reason": f"parse error: {e}"}
        res.setdefault("mode", mode)
        per_mode.append(res)

    strict = next((m for m in per_mode if m.get("mode") == "strict"), {})
    missing = len(strict.get("missing_agents") or [])
    extra = len(strict.get("extra_agents") or [])
    denom = len(expected_names) + len(actual_names)
    if strict.get("match") == "Pass":
        score = 10
    elif (missing + extra) == 0:
        # same agent multiset, order differs (or no diff detail) -> partial credit
        score = 6
    else:
        score = _score_from_ratio(max(0, denom - (missing + extra)), denom) or 1

    reasoning = ""
    try:
        summary = [{"mode": m.get("mode"), "match": m.get("match"), "reason": m.get("reason")}
                   for m in per_mode]
        rr = invoke_llm(
            NATIVE_PROMPTS["trajectory_match"]["overall_reason"]
            .replace("{PER_MODE_SUMMARY}", json.dumps(summary, indent=2)),
            "Return the JSON object now.",
        )
        reasoning = parse_json(rr).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    miss = strict.get("missing_agents") or []
    ext = strict.get("extra_agents") or []
    if miss:
        reasoning += _detail_suffix("Missing steps", miss, limit=6)
    if ext:
        reasoning += _detail_suffix("Extra/unexpected steps", ext, limit=6)
    if not miss and not ext and strict.get("match") != "Pass":
        reasoning += " Same steps as expected but executed in a different order."
    return {"metric": "trajectory_match", "score": score, "max_score": 10,
            "reasoning": reasoning,
            "raw_evaluation": json.dumps(per_mode, default=str)[:4000]}


def run_plan_accuracy(span, execution_trace):
    if not span:
        return _missing("plan_accuracy", "Supervisor: Plan & Route span not found.")
    plan_inputs = span_inputs(span)
    supervisor_rules = (plan_inputs.get("instruction")
                        or json.dumps(plan_inputs, indent=2, default=str))
    ap = (
        NATIVE_PROMPTS["plan_accuracy"]["audit"]
        .replace("{USER_INPUT}", json.dumps(execution_trace.get("inputs", {}), indent=2, default=str))
        .replace("{RULES}", supervisor_rules)
        .replace("{PLAN}", json.dumps(span_outputs(span), indent=2, default=str))
    )
    raw = invoke_llm(ap, "Return the strict JSON now.")
    parsed = parse_json(raw)
    frags = parsed.get("fragments", []) if isinstance(parsed, dict) else parsed
    passed = sum(1 for f in frags
                 if f.get("Plan Output Rule Accuracy") == "pass"
                 and f.get("Plan Output Flow Accuracy") == "pass")
    score = _score_from_ratio(passed, len(frags))
    reasoning = ""
    try:
        rr = invoke_llm(
            NATIVE_PROMPTS["plan_accuracy"]["overall_reason"]
            .replace("{FRAGMENT_RESULTS}", json.dumps(frags, indent=2, default=str)),
            "Return the JSON object now.",
        )
        reasoning = parse_json(rr).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    failing = [f.get("Output Fragment", "?") for f in frags
               if f.get("Plan Output Rule Accuracy") != "pass"
               or f.get("Plan Output Flow Accuracy") != "pass"]
    if failing:
        reasoning += f" {len(failing)}/{len(frags)} plan steps failed a rule or flow check."
        reasoning += _detail_suffix("Failed steps", failing, limit=3)
    elif frags:
        reasoning += f" All {len(frags)} plan steps passed rule and flow checks."
    return {"metric": "plan_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning, "raw_evaluation": raw}


def run_tool_call_accuracy(expected_tools, tool_spans):
    if not tool_spans:
        return _missing("tool_call_accuracy", "No tool/sub-agent spans found in trace.")
    actual = [{"tool_name": s["name"], "parameters": span_inputs(s)} for s in tool_spans]
    raw = invoke_llm(TOOL_CALL_ACCURACY_SYSTEM_PROMPT,
                     build_tool_call_accuracy_user_prompt(expected_tools, actual))
    parsed = parse_json(raw)
    tools = parsed.get("tools", []) if isinstance(parsed, dict) else parsed
    passed = sum(1 for t in tools
                 if t.get("tool_name_check") == "pass"
                 and t.get("tool_parameter_check") == "pass"
                 and t.get("parameter_format_check") == "pass")
    score = _score_from_ratio(passed, len(tools))
    reasoning = parsed.get("overall_reason", "") if isinstance(parsed, dict) else ""
    failing_tools = [t.get("tool_name", "?") for t in tools
                     if t.get("tool_name_check") != "pass"
                     or t.get("tool_parameter_check") != "pass"
                     or t.get("parameter_format_check") != "pass"]
    if failing_tools:
        reasoning += _detail_suffix("Tool calls failing a check", failing_tools, limit=6)
    elif tools:
        reasoning += f" All {len(tools)} tool calls conform to the expected schema."
    return {"metric": "tool_call_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning, "raw_evaluation": raw}


def run_summary_accuracy(spans):
    if not spans:
        return _missing("summary_accuracy", "No summary-producing spans found.")
    claim_records, total, matched = [], 0, 0
    for span in spans:
        summary_text = json.dumps(span_outputs(span), default=str)
        try:
            claims = parse_json(invoke_llm(
                NATIVE_PROMPTS["summary_accuracy"]["claim_segmentation"].replace("{SUMMARY}", summary_text),
                "Return the JSON object now.",
            )).get("claims", [])
        except Exception as e:
            claim_records.append({"agent": span["name"], "claim": "(segmentation failed)",
                                  "status": "No Match", "evidence": "", "reason": str(e)})
            continue
        context = collect_evidence_context(span)
        for claim in claims:
            try:
                v = parse_json(invoke_llm(
                    NATIVE_PROMPTS["summary_accuracy"]["claim_verification"],
                    json.dumps({"Output Fragment": claim, "context": context}),
                ))
                status = v.get("Evidence Match Status", "No Match")
                evidence = v.get("Supporting Context Evidence", "")
                reason = v.get("Verification Reason", "")
            except Exception as e:
                status, evidence, reason = "No Match", "", f"verify error: {e}"
            total += 1
            matched += 1 if status == "Match" else 0
            claim_records.append({"agent": span["name"], "claim": claim, "status": status,
                                  "evidence": evidence, "reason": reason})
    score = _score_from_ratio(matched, total)
    reasoning = (f"{matched}/{total} summary claims supported by trace context."
                 if total else "No verifiable factual claims were segmented.")
    unsupported = [c.get("claim", "?") for c in claim_records if c.get("status") != "Match"]
    if unsupported:
        reasoning += _detail_suffix("Unsupported claims", unsupported, limit=4)
    return {"metric": "summary_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning,
            "raw_evaluation": json.dumps(claim_records, default=str)[:4000]}


def run_goal_accomplishment(execution_trace):
    raw = invoke_llm(
        build_goal_accomplishment_prompt(
            json.dumps((execution_trace.get("inputs") or {}).get(
                "user_request", execution_trace.get("inputs", {})), default=str),
            json.dumps(execution_trace.get("outputs", {}), default=str),
        ),
        "Return the JSON object now.",
    )
    parsed = parse_json(raw)
    try:
        score = max(1, min(10, round(float(parsed.get("verdict", 0)) * 10)))
    except (TypeError, ValueError):
        score = None
    goal_fragments = parsed.get("fragments", []) or []
    reasoning = parsed.get("reason", "")
    unmet = [f.get("query_fragment", "?") for f in goal_fragments
             if f.get("status") != "Match"]
    if unmet:
        reasoning += _detail_suffix("Unaddressed requirements", unmet, limit=6)
    return {"metric": "goal_accomplishment", "score": score, "max_score": 10,
            "reasoning": reasoning,
            "raw_evaluation": raw}


# ------------------------------------------------------------
# Metric targets + dispatcher (same interface as before)
# ------------------------------------------------------------
def build_metric_targets(execution_trace, trace_data, expected):
    sl = trace_data["span_lookup"]
    return {
        "trajectory_match": {"expected": expected["trajectory"], "actual": execution_trace},
        "parameter_extraction": {"actual": first_span(sl, "Parameter Extraction Agent")},
        "plan_accuracy": {"actual": first_span(sl, "Supervisor: Plan & Route")},
        "tool_call_accuracy": {"expected": expected["tools"],
                               "actual": trace_data["tool_spans"]},
        # summary_accuracy audits the FINAL report (the summary of record),
        # verified against the gathered research/analysis/regulatory evidence.
        "summary_accuracy": {"actual": sl.get("Report Writer Agent", [])},
        "goal_accomplishment": {"actual": first_span(sl, "Supervisor: Final Review")},
    }


def evaluate_metric(metric_name, metric_targets, execution_trace):
    t = metric_targets[metric_name]
    try:
        if metric_name == "parameter_extraction":
            return run_parameter_extraction(t["actual"])
        if metric_name == "trajectory_match":
            return run_trajectory_match(t["expected"], t["actual"])
        if metric_name == "plan_accuracy":
            return run_plan_accuracy(t["actual"], execution_trace)
        if metric_name == "tool_call_accuracy":
            return run_tool_call_accuracy(t["expected"], t["actual"])
        if metric_name == "summary_accuracy":
            return run_summary_accuracy(t["actual"])
        if metric_name == "goal_accomplishment":
            return run_goal_accomplishment(execution_trace)
    except (ValueError, json.JSONDecodeError) as error:
        return {"metric": metric_name, "score": None, "max_score": 10,
                "reasoning": f"Evaluator response invalid: {error}",
                "parse_error": str(error)}
    raise ValueError(f"Unsupported metric: {metric_name}")


print("Native multi-stage evaluation working loaded for all six metrics.")


Native multi-stage evaluation working loaded for all six metrics.


## 9. Evaluate and Export Results

Runs every selected metric for every retrieved trace. Results are tagged with trace and variant metadata, then written to `evaluation_results.json` and `evaluation_results.xlsx`. The final output prints an average score per trace.


In [ ]:
# ============================================================
# Execute Evaluation Pipeline for All Retrieved Traces
# ============================================================

def get_variant_id(trace_name):
    """Map the logger's root-run name to an expected-result variant ID."""
    prefix = "Supervisor Agent — "
    return trace_name.removeprefix(prefix) if trace_name.startswith(prefix) else "baseline"


evaluation_results = []

print("=" * 70)
print("Running Evaluation Pipeline")
print("=" * 70)

for trace_number, (root_run, execution_trace) in enumerate(
    zip(root_runs, execution_traces), start=1
):
    print(f"\nTrace {trace_number}/{len(root_runs)}: {root_run.name} ({root_run.id})")

    expected = resolve_expected_config(execution_trace)
    trace_data = preprocess_execution_trace(execution_trace, set(expected["tools"].keys()))
    metric_targets = build_metric_targets(execution_trace, trace_data, expected)

    for metric in SELECTED_METRICS:
        print(f"  Evaluating: {metric}")
        try:
            result = evaluate_metric(metric, metric_targets, execution_trace)
        except Exception as error:
            result = {
                "metric": metric,
                "score": None,
                "max_score": 10,
                "reasoning": f"Evaluation failed: {error}",
            }

        result.update({
            "variant_id": get_variant_id(root_run.name),
            "trace_id": str(root_run.id),
            "trace_name": root_run.name,
            "trace_start_time": (
                root_run.start_time.isoformat() if root_run.start_time else None
            ),
            "lookback_hours": LOOKBACK_HOURS,
        })
        evaluation_results.append(result)

print("=" * 70)
print("Evaluation Completed")
print("=" * 70)
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")


# ============================================================
# Process Evaluation Results
# ============================================================

processed_results = [
    {
        "Variant": result.get("variant_id"),
        "Trace ID": result.get("trace_id"),
        "Trace Name": result.get("trace_name"),
        "Trace Start Time": result.get("trace_start_time"),
        "Metric": result.get("metric"),
        "Score": result.get("score"),
        "Max Score": result.get("max_score"),
        "Reasoning": result.get("reasoning"),
        "Recommendation": result.get("recommendation"),
    }
    for result in evaluation_results
]

print(f"Processed {len(processed_results)} metric result(s).")


# ============================================================
# Export Results to JSON
# ============================================================

JSON_OUTPUT_FILE = "evaluation_results.json"

with open(JSON_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=4, ensure_ascii=False)

print(f"✓ JSON results exported to '{JSON_OUTPUT_FILE}'")

# ============================================================
# Export Results to Excel
# ============================================================

EXCEL_OUTPUT_FILE = "evaluation_results.xlsx"

df = pd.DataFrame(processed_results)
df.to_excel(EXCEL_OUTPUT_FILE, index=False)

print(f"✓ Excel results exported to '{EXCEL_OUTPUT_FILE}'")


# ============================================================
# Evaluation Summary
# ============================================================

print("=" * 70)
print("LangSmith Trace Evaluation Completed")
print("=" * 70)
print(f"Lookback window   : {LOOKBACK_HOURS} hour(s)")
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")
print(f"JSON Report       : {JSON_OUTPUT_FILE}")
print(f"Excel Report      : {EXCEL_OUTPUT_FILE}")
print("=" * 70)

summary_df = pd.DataFrame(evaluation_results)
summary_df["score"] = pd.to_numeric(summary_df["score"], errors="coerce")

print("\nAverage score by trace:")
for trace_name, group in summary_df.groupby("trace_name", dropna=False):
    average = group["score"].mean()
    score_text = f"{average:.2f}/10" if pd.notna(average) else "N/A"
    print(f"- {trace_name}: {score_text}")

print("\nEvaluation completed successfully.")


Running Evaluation Pipeline

Trace 1/5: Supervisor Agent (019f7dcf-a33a-7011-8e65-260f2b7f36f3)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 2/5: Supervisor Agent — variant_A_poor_parameter_and_plan (019f7dcf-a46c-7d30-9fa3-6a41900595ba)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 3/5: Supervisor Agent — variant_B_poor_trajectory_and_tools (019f7dcf-a4da-7a81-b7c6-098bb26af9f1)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 4/5: Supervisor Agent — variant_C_poor_summary_accuracy (019f7dcf-a558-7d01-aa95-2ac0ca689a73)
  Evaluating: tr

## 11. Log Evaluation Results back to LangSmith

Logs the evaluation results as feedback into LangSmith.
- **Trace-level metrics** (`goal_accomplishment`, `trajectory_match`) → attached to root trace
- **Span-level metrics** (`parameter_extraction`, `plan_accuracy`, `tool_call_accuracy`, `summary_accuracy`) → attached to their respective child spans


In [ ]:
# ============================================================
# Log Evaluation Results back to LangSmith
# ============================================================

from langsmith import Client

def get_span_id_by_name(execution_trace, span_name):
    """Recursively find the first span with the given name and return its id."""
    if execution_trace.get("name") == span_name:
        return execution_trace.get("id")
    for child in execution_trace.get("children", []):
        found = get_span_id_by_name(child, span_name)
        if found:
            return found
    return None


def log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces):
    """Log evaluation metrics back to LangSmith as feedback."""
    print("=" * 70)
    print("Logging Evaluation Results to LangSmith")
    print("=" * 70)

    # Group results by trace
    results_by_trace = {}
    for res in evaluation_results:
        tid = res.get("trace_id")
        if tid not in results_by_trace:
            results_by_trace[tid] = []
        results_by_trace[tid].append(res)

    logged_count = 0

    for idx, (root_run, exec_trace) in enumerate(zip(root_runs, execution_traces)):
        trace_id = str(root_run.id)
        trace_results = results_by_trace.get(trace_id, [])

        if not trace_results:
            continue

        print(f"\nLogging feedback for trace {idx+1}: {root_run.name}")

        for result in trace_results:
            metric = result.get("metric")
            score = result.get("score")
            reasoning = result.get("reasoning", "")

            if score is None:
                continue

            # Normalize score to 0-1 range for LangSmith (optional but common)
            normalized_score = round(score / 10.0, 4)

            # ============================================================
            # TRACE-LEVEL METRICS → attach to root
            # ============================================================
            if metric in ["goal_accomplishment", "trajectory_match"]:
                try:
                    langsmith_client.create_feedback(
                        key=metric,
                        score=normalized_score,
                        trace_id=trace_id,
                        comment=reasoning[:500] if reasoning else None,
                        source_info={
                            "evaluator": "llm-judge",
                            "raw_score": score,
                            "max_score": 10
                        }
                    )
                    print(f"  ✓ Trace-level: {metric} = {score}/10 → root trace")
                    logged_count += 1
                except Exception as e:
                    print(f"  ✗ Failed to log {metric}: {e}")

            # ============================================================
            # SPAN-LEVEL METRICS → attach to specific spans
            # ============================================================
            elif metric == "parameter_extraction":
                span_id = get_span_id_by_name(exec_trace, "Parameter Extraction Agent")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="parameter_extraction_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: parameter_extraction → Parameter Extraction Agent")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "plan_accuracy":
                span_id = get_span_id_by_name(exec_trace, "Supervisor: Plan & Route")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="plan_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "tool_call_accuracy":
                # Attach to Research Agent and Analysis Agent as examples
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"tool_call_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: tool_call_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

            elif metric == "summary_accuracy":
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"summary_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: summary_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

    print("=" * 70)
    print(f"Successfully logged {logged_count} feedback entries to LangSmith.")
    print("=" * 70)


# Run the logging
log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces)


Logging Evaluation Results to LangSmith

Logging feedback for trace 1: Supervisor Agent
  ✓ Trace-level: trajectory_match = 10/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis Agent
  ✓ Span-level: tool_call_accuracy → Report Writer Agent
  ✓ Span-level: summary_accuracy → Research Agent
  ✓ Span-level: summary_accuracy → Analysis Agent
  ✓ Span-level: summary_accuracy → Report Writer Agent
  ✓ Trace-level: goal_accomplishment = 10/10 → root trace

Logging feedback for trace 2: Supervisor Agent — variant_A_poor_parameter_and_plan
  ✓ Trace-level: trajectory_match = 10/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis 